# 04 — ChromaDB with custom Gemini embeddings

Create a persistent ChromaDB collection and insert documents using embedding vectors generated by the Gemini embedding model (`gemini-embedding-001`).

In [ ]:
!pip install chromadb google-genai python-dotenv

In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from google import genai
import chromadb
from chromadb import Documents, EmbeddingFunction, Embeddings

load_dotenv(find_dotenv())
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

## Custom embedding function

ChromaDB calls this function whenever documents (or queries) need to be turned into vectors.

In [ ]:
class GeminiEmbeddingFunction(EmbeddingFunction):
    def __call__(self, input: Documents) -> Embeddings:
        response = gemini_client.models.embed_content(
            model="gemini-embedding-001",
            contents=input,
        )
        return [embedding.values for embedding in response.embeddings]


gemini_ef = GeminiEmbeddingFunction()

## Create persistent collection and insert documents

Embeddings are generated with Gemini and stored in the vector DB.

In [ ]:
client = chromadb.PersistentClient(path="chromadb_gemini")
collection = client.create_collection(
    name="chromadb_gemini",
    embedding_function=gemini_ef,
)

documents = [
    "El costo base de la cuenta es 1000.",
    "El costo por consumo es 200.",
    "El costo total de una cuenta se calcula sumando el costo base y el costo por consumo.",
    "{'producto':'modem', 'costo':'4000'}",
    "{'producto':'cable ethernet', 'costo':'500'}",
    "{'producto':'regulador', 'costo':'3000'}",
    "{'producto':'repetidor', 'costo':'3500'}",
]
ids = ["id1", "id2", "id3", "id4", "id5", "id6", "id7"]

collection.add(
    documents=documents,
    ids=ids,
)

print(f"Collection count: {collection.count()}")
print(f"Embedding dimension: {len(gemini_ef(documents[:1])[0])}")

## Query with the same custom embeddings

In [ ]:
#prompt = "cuanto cuesta un regulador?"
prompt = "cuanto tengo que pagar si compro dos reguladores y dos modems"

results = collection.query(
    query_texts=[prompt],
    n_results=3,
)

print("Query:", prompt)
print("Relevant documents:")
for doc, doc_id, distance in zip(
    results["documents"][0],
    results["ids"][0],
    results["distances"][0],
):
    print(f"  [{doc_id}] (distance={distance:.4f}) {doc}")

## Answer with Gemini LLM

Pass the retrieved context and the user prompt to the model.

In [ ]:
context = "\n".join(results["documents"][0])

contents = f"""Usa solo el siguiente contexto para responder la pregunta.
Si un documento del contexto no es relevante, ignóralo.
Si el contexto no contiene la respuesta, di que no lo sabes.

Contexto:
{context}

Pregunta: {prompt}
"""

response = gemini_client.models.generate_content(
    model="gemini-3.6-flash",
    contents=contents,
)

# Prefer text parts only (avoids thought_signature warning on thinking models)
answer = "".join(
    part.text
    for part in response.candidates[0].content.parts
    if getattr(part, "text", None)
)
print(answer)